# 02 — Forward diffusion process (Phase 1.1)

Apply the closed-form forward process to a Pokémon sprite and watch it dissolve into noise across `T = 1000` timesteps.

$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import torch
from matplotlib import pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import PokemonDataset

%matplotlib inline
torch.manual_seed(42)

## 2. Load one Pokémon

Bulbasaur as the test image. Small helper to display a `[-1, 1]` tensor (clip + permute + rescale to `[0, 1]`).

In [ ]:
ds = PokemonDataset(
    csv_path=PROJECT_ROOT / "data" / "metadata.csv",
    images_dir=PROJECT_ROOT / "data" / "images",
)
x_0, label = ds[0]
print(f"Shape : {x_0.shape}, dtype : {x_0.dtype}")
print(f"Range : [{x_0.min():.2f}, {x_0.max():.2f}]")


def show_tensor(t, ax=None, title=""):
    img = t.clamp(-1, 1).permute(1, 2, 0).numpy()
    img = (img + 1) / 2
    if ax is None:
        _, ax = plt.subplots()
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title)


show_tensor(x_0, title="x_0")
plt.show()

## 3. Noise schedule

Linear schedule for `β_t` (DDPM default). From there: `α_t = 1 - β_t`, `ᾱ_t = ∏_{i ≤ t} α_i`.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

print(f"betas[0]           = {betas[0]:.6f}")
print(f"betas[-1]          = {betas[-1]:.6f}")
print(f"alphas_cumprod[0]  = {alphas_cumprod[0]:.6f}")
print(f"alphas_cumprod[-1] = {alphas_cumprod[-1]:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].plot(betas);          axes[0].set_title("betas")
axes[1].plot(alphas);         axes[1].set_title("alphas")
axes[2].plot(alphas_cumprod); axes[2].set_title("alphas_cumprod")
plt.tight_layout()
plt.show()

## 4. Sanity checks

`β` strictly increasing, `ᾱ` strictly decreasing, `ᾱ_{T-1}` close to zero (so `x_T ≈ N(0, I)`).

In [ ]:
assert (betas.diff() > 0).all(), "betas should be strictly increasing"
assert (alphas_cumprod.diff() < 0).all(), "alphas_cumprod should be strictly decreasing"
print(f"alphas_cumprod[-1] = {alphas_cumprod[-1]:.2e}")

## 5. Closed-form forward process

$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \varepsilon$$

Lets us jump to any `t` in one step instead of iterating from `x_0`.

In [ ]:
def q_sample(x_0, t, noise):
    """x_t = sqrt(ᾱ_t) * x_0 + sqrt(1 - ᾱ_t) * noise."""
    ac = alphas_cumprod[t]
    return ac.sqrt() * x_0 + (1 - ac).sqrt() * noise


noise = torch.randn_like(x_0)
x_500 = q_sample(x_0, t=500, noise=noise)
print(f"x_500 shape : {x_500.shape}")
print(f"x_500 range : [{x_500.min():.2f}, {x_500.max():.2f}]")

## 6. Forward process at increasing `t`

In [ ]:
timesteps = [0, 50, 100, 200, 400, 700, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(2 * len(timesteps), 2.5))

for ax, t in zip(axes, timesteps):
    noise = torch.randn_like(x_0)
    x_t = q_sample(x_0, t, noise)
    show_tensor(x_t, ax=ax, title=f"t={t}")

plt.tight_layout()
plt.show()

## 7. Observations

The image stays clearly recognizable up to `t ≈ 100`, degrades visibly between `t = 200` and `t = 400`, and is essentially pure noise from `t ≈ 500` onwards.

Half the timesteps (≈ `t > 500`) look indistinguishable from `x_T`. With a linear schedule, a lot of the chain is spent on already-saturated noise — the model would waste capacity learning to denoise indistinguishable inputs. This is the motivation for the **cosine schedule** (Nichol & Dhariwal, 2021) we'll implement next in `src/diffusion/schedule.py`.